# Notebook 14: The Feature Channel — Does Query-Conditional Selection Help?

**Purpose**: Test the one channel Notebooks 11-13 left open. Produces the
project's first positive protocol result, and two constraints on how it may be
reported.

## The logic of the test

Notebook 13 established that these models do not learn the input→label mapping
from tabular demonstrations on 3 of 4 datasets, at either scale. That closes the
**label channel** — but not the **feature channel**: query-conditional selection
works by putting *feature values relevant to this query* into the prompt, which
does not require the labels to be informative at all.

So run every mechanism at **0% and 100% label corruption**. With every label
flipped, any remaining advantage over random-k cannot come from the label
mapping. It must be feature content. This decomposes the two channels directly
rather than inferring one from the other.

```bash
cd sata-project
# 8B, calibration ON (Notebook 13 §4)
PYTHONPATH=. python scripts/run_real_arm_grid.py --mode grid \
    --cache-dir _screen_cache --model Llama-3.1-8B-Instruct \
    --out results/v2/feature_channel_8b \
    --datasets anes acsincome brfss_diabetes \
    --mechanisms random similarity feature_coverage importance_weighted \
    --composition balanced --corruptions 0.0 1.0 \
    --seeds 42 123 456 789 1024 2048 4096 8192 --query-split ood \
    --tensor-parallel 1 --max-model-len 4096
# 70B fp8, calibration OFF -- sign-inverted there, and skipping it halves cost
#   ... --no-calibration --quantization fp8 --shard 0 --n-shards 2
```

**Design choices and why.** `acspubcov` is excluded — Notebooks 10, 11 and 13 all
independently put it at chance, so it is a documented negative control rather
than a test case. Composition is held at `balanced` because Notebook 11 showed
demo label counts do not drive the output prior and Notebook 13 found no
systematic composition effect; sweeping it three ways would triple cost for a
factor already measured as null. **Eight seeds** instead of five, for the reason
in §4. The ID split is dropped because Notebook 13 measured the gap at ~0.

195 units / 48,750 rows per scale.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

pd.set_option("display.width", 220)
RESULTS = PROJECT_ROOT / "results" / "v2"


def margin(df):
    """Label-logprob margin: the model's decision variable before thresholding."""
    return df.logprob_1 - df.logprob_0


def p_positive(df):
    """Implied P(positive) = sigmoid(margin)."""
    return 1.0 / (1.0 + np.exp(df.logprob_0 - df.logprob_1))


def integrity(df, name=""):
    """A failed label-token lookup defaults to -100 and a failed decode to -1.
    Both would masquerade as findings, so check before interpreting anything."""
    bad_lp = ((df.logprob_0 == -100) | (df.logprob_1 == -100)).mean()
    bad_pred = (df.prediction_raw == -1).mean()
    print(f"{name}invalid predictions: {bad_pred:.5f} | logprob sentinels: {bad_lp:.5f}")
    return bad_lp == 0 and bad_pred == 0

## Step 0: Load, and confirm the mechanism does what it claims

Before interpreting anything: `similarity` is only meaningful if it actually
selects *different demonstrations for different queries*. `demo_ids` is recorded
per row, so this is checkable directly rather than assumed. The three fixed-set
mechanisms should show exactly one distinct set per unit.

In [3]:
import json

F8 = pd.read_parquet(RESULTS / "feature_channel_8b_merged.parquet")
F70 = pd.read_parquet(RESULTS / "feature_channel_70b_merged.parquet")
F = pd.concat([F8.assign(scale="8B"), F70.assign(scale="70B")], ignore_index=True)
assert integrity(F8, "8B: ") and integrity(F70, "70B: ")

unit_keys = ["dataset", "mechanism", "composition", "corruption", "seed"]
probe = (F8[F8.mechanism != "zero_shot"].groupby(unit_keys)
           .demo_ids.apply(lambda s: s.map(lambda d: tuple(sorted(json.loads(d)))).nunique()))
print("distinct demo sets per unit (250 queries each):")
print(probe.groupby(level="mechanism").agg(["min", "max"]).to_string())

8B: invalid predictions: 0.00000 | logprob sentinels: 0.00000
70B: invalid predictions: 0.00000 | logprob sentinels: 0.00000
distinct demo sets per unit (250 queries each):
                     min  max
mechanism                    
feature_coverage       1    1
importance_weighted    1    1
random                 1    1
similarity           231  249


`similarity` varies its selection across essentially every query; the other
three reuse one set per unit, as designed. The mechanism is doing what its name
says.

## Step 1: Per-seed metrics

AUROC on the raw label-logprob margin is primary — threshold-free, so immune to
the scale-dependent calibration pathology of Notebook 13 §4. Balanced accuracy
uses **calibrated** predictions at 8B and **raw** at 70B, per that same finding.

In [4]:
rows = []
for (sc, ds, mech, corr), g in F[F.mechanism != "zero_shot"].groupby(
        ["scale", "dataset", "mechanism", "corruption"]):
    for s, x in g.groupby("seed"):
        pred = x.prediction if sc == "8B" else x.prediction_raw
        rows.append(dict(scale=sc, dataset=ds, mechanism=mech, corruption=corr, seed=s,
                         auroc=roc_auc_score(x.label, margin(x)),
                         bacc=balanced_accuracy_score(x.label, pred)))
M = pd.DataFrame(rows)
assert len(M) == 2 * 3 * 4 * 2 * 8, len(M)

M.pivot_table(index=["scale", "dataset"], columns=["mechanism", "corruption"],
              values="auroc").round(3)

mechanism            feature_coverage        importance_weighted        random        similarity       
corruption                        0.0    1.0                 0.0    1.0    0.0    1.0        0.0    1.0
scale dataset                                                                                          
70B   acsincome                 0.845  0.831               0.836  0.826  0.839  0.826      0.821  0.786
      anes                      0.716  0.599               0.735  0.611  0.735  0.611      0.667  0.476
      brfss_diabetes            0.784  0.742               0.768  0.744  0.778  0.722      0.756  0.673
8B    acsincome                 0.702  0.677               0.686  0.641  0.677  0.667      0.718  0.700
      anes                      0.669  0.504               0.690  0.498  0.690  0.498      0.619  0.591
      brfss_diabetes            0.652  0.610               0.610  0.604  0.608  0.612      0.716  0.730

## Step 2: The pre-registered test

Every mechanism against the `random` baseline, paired by seed, at each corruption
level. The column that matters is **corruption = 1.0**: a positive delta there is
feature-channel evidence.

In [5]:
base = M[M.mechanism == "random"].set_index(["scale", "dataset", "corruption", "seed"]).auroc
res = []
for (sc, ds, mech, corr), g in M[M.mechanism != "random"].groupby(
        ["scale", "dataset", "mechanism", "corruption"]):
    g = g.sort_values("seed")
    b = base.loc[(sc, ds, corr)].reindex(g.seed.values).to_numpy()
    a = g.auroc.to_numpy()
    res.append(dict(scale=sc, dataset=ds, mechanism=mech, corruption=corr,
                    delta=(a - b).mean(), seeds_pos=int((a > b).sum()),
                    p=stats.wilcoxon(a, b).pvalue if np.any(a != b) else 1.0,
                    identical=bool(np.all(a == b))))
Rv = pd.DataFrame(res)
Rv.pivot_table(index=["scale", "dataset", "mechanism"], columns="corruption",
               values=["delta", "seeds_pos", "p"]).round(3)

delta             p        seeds_pos     
corruption                                  0.0    1.0    0.0    1.0       0.0  1.0
scale dataset        mechanism                                                     
70B   acsincome      feature_coverage     0.006  0.005  0.109  0.461       5.0  4.0
                     importance_weighted -0.003  0.000  0.461  0.844       3.0  4.0
                     similarity          -0.018 -0.041  0.109  0.008       3.0  0.0
      anes           feature_coverage    -0.019 -0.012  0.461  0.641       4.0  4.0
                     importance_weighted  0.000  0.000  1.000  1.000       0.0  0.0
                     similarity          -0.068 -0.135  0.008  0.008       0.0  0.0
      brfss_diabetes feature_coverage     0.006  0.021  0.844  0.383       4.0  5.0
                     importance_weighted -0.010  0.022  0.312  0.312       2.0  5.0
                     similarity          -0.022 -0.049  0.109  0.016       2.0  1.0
8B    acsincome      feature_coverage     0.025  0.011  0.312  0.641       4.0  4.0
                     importance_weighted  0.008 -0.026  0.641  0.195       4.0  3.0
                     similarity           0.041  0.033  0.023  0.023       7.0  7.0
      anes           feature_coverage    -0.021  0.006  0.312  0.312       2.0  6.0
                     importance_weighted -0.000  0.000  1.000  1.000       0.0  0.0
                     similarity          -0.071  0.093  0.039  0.016       2.0  7.0
      brfss_diabetes feature_coverage     0.044 -0.002  0.055  0.742       6.0  4.0
                     importance_weighted  0.002 -0.007  0.945  0.844       3.0  3.0
                     similarity           0.108  0.119  0.039  0.008       6.0  8.0

## Step 3: Multiple comparisons, and why the family matters

Two facts have to be reported together.

**`importance_weighted` is bit-identical to `random` on ANES** at both scales.
That is the Notebook 09 degraded fallback firing exactly as designed — ANES's
domain-discriminator AUC is 0.557, below the 0.60 detectability threshold, so the
density ratio is noise and the mechanism correctly declines to use it. Those cells
carry no test and are excluded rather than counted as null results.

In [6]:
from statsmodels.stats.multitest import multipletests

live = Rv[~Rv.identical].copy()
rej, p_adj, _, _ = multipletests(live.p.values, alpha=0.05, method="holm")
live["p_holm_full"], live["sig_full"] = p_adj, rej

floor8 = stats.wilcoxon(np.arange(1, 9) * 1.0, np.zeros(8)).pvalue
print(f"testable comparisons: {len(live)} (excluded {Rv.identical.sum()} degraded-fallback cells)")
print(f"survive Holm over the full grid: {int(rej.sum())}")
print(f"\nWilcoxon floor at 8 seeds = {floor8:.4f}")
print(f"Holm threshold for {len(live)} tests = {0.05/len(live):.5f}  -> unsatisfiable")
nmin = next(n for n in range(4, 25)
            if stats.wilcoxon(np.arange(1, n+1) * 1.0, np.zeros(n)).pvalue < 0.05 / len(live))
print(f"seeds needed for a full-grid correction to be satisfiable: {nmin}")

testable comparisons: 33 (excluded 3 degraded-fallback cells)
survive Holm over the full grid: 0

Wilcoxon floor at 8 seeds = 0.0078
Holm threshold for 33 tests = 0.00152  -> unsatisfiable
seeds needed for a full-grid correction to be satisfiable: 11


**Zero of the exploratory comparisons survive Holm — and that is arithmetic, not
weak effects.** The smallest *p* an 8-seed Wilcoxon can return is 0.0078, while
Holm over 33 tests demands 0.0015. No effect size whatsoever could have passed.

This is why the confirmatory claim is restricted to the **pre-registered family**
(`GATE_S0C_FINDINGS.md` §10.7 named `similarity`-vs-random-at-100%-corruption as
*the* test), with everything else exploratory. That distinction is load-bearing,
not presentational.

In [7]:
sim = live[(live.corruption == 1.0) & (live.mechanism == "similarity")].copy()
r3, p3, _, _ = multipletests(sim.p.values, alpha=0.05, method="holm")
sim["p_holm"], sim["significant"] = p3, r3
sim.sort_values("p")[["scale", "dataset", "delta", "seeds_pos", "p", "p_holm", "significant"]].round(4)

,scale,dataset,delta,seeds_pos,p,p_holm,significant
5,70B,acsincome,-0.0407,0,0.0078,0.0469,True
11,70B,anes,-0.1348,0,0.0078,0.0469,True
35,8B,brfss_diabetes,0.1189,8,0.0078,0.0469,True
17,70B,brfss_diabetes,-0.0488,1,0.0156,0.0469,True
29,8B,anes,0.0931,7,0.0156,0.0469,True
23,8B,acsincome,0.0333,7,0.0234,0.0469,True


**All six significant.** With every demonstration label wrong:

- **8B**: similarity beats random-k on all three datasets (+0.119 BRFSS with 8/8
  seeds, +0.093 ANES, +0.033 ACS Income). **The feature channel is real.** This is
  the project's first positive protocol result.
- **70B**: similarity *loses* on all three (−0.135 ANES, −0.049 BRFSS, −0.041 ACS
  Income), also significantly. **The effect inverts with scale.**

## Step 4: Label-robustness confirms the mechanism

Not just *whether* similarity helps, but whether it helps *for the stated
reason*. A feature-channel mechanism should be relatively indifferent to label
corruption.

In [8]:
rob = (M.pivot_table(index=["scale", "dataset", "mechanism"], columns="corruption", values="auroc")
         .assign(loss=lambda d: d[0.0] - d[1.0]))
rob.reset_index().pivot_table(index=["scale", "dataset"], columns="mechanism",
                              values="loss").round(3)

mechanism             feature_coverage  importance_weighted  random  similarity
scale dataset                                                                  
70B   acsincome                  0.014                0.009   0.013       0.035
      anes                       0.117                0.124   0.124       0.191
      brfss_diabetes             0.041                0.024   0.056       0.083
8B    acsincome                  0.025                0.045   0.010       0.018
      anes                       0.165                0.192   0.192       0.028
      brfss_diabetes             0.042                0.006  -0.004      -0.015

At 8B, `similarity` on ANES loses **0.028** AUROC to full label corruption where
`random` loses **0.192** — it substitutes feature information for the label
information it no longer has. On BRFSS its loss is slightly *negative*: marginally
better with the labels destroyed. At 70B the ordering **reverses** on all three
datasets — similarity becomes *more* label-dependent than random.

This also explains the odd ANES 8B sign flip in Step 2 (similarity is worse at 0%
corruption, −0.071, but better at 100%, +0.093). ANES is the one dataset where the
label channel is live (Notebook 13 §3), so with correct labels the label channel
dominates and similarity's neighbour-constrained label balance costs more than its
feature gain; with labels destroyed only the feature channel remains.

**An untested hypothesis for the 70B inversion**, flagged as such: the k=8 nearest
neighbours are a low-diversity, locally biased sample, and a model with strong
enough priors may do better from a diverse random draw than a narrow local one.
Testing it needs a diversity-controlled variant (k-medoids within the
neighbourhood), which is not in this run.

## Step 5: The reality check — the baseline was weak

In [9]:
zs = (F[F.mechanism == "zero_shot"].groupby(["scale", "dataset"])
        .apply(lambda x: roc_auc_score(x.label, margin(x))).rename("zero_shot"))
zrows = []
for (sc, ds, mech), g in M[M.corruption == 0.0].groupby(["scale", "dataset", "mechanism"]):
    z, a = zs.loc[(sc, ds)], g.auroc.to_numpy()
    zrows.append(dict(scale=sc, dataset=ds, mechanism=mech, delta_vs_zero_shot=a.mean() - z,
                      seeds_above=int((a > z).sum()), p=stats.wilcoxon(a - z).pvalue))
Zt = pd.DataFrame(zrows)
print("cells beating a zero-shot prompt (uncorrected):")
print(Zt[(Zt.delta_vs_zero_shot > 0) & (Zt.p < 0.05)].round(4).to_string(index=False))
print(f"\n{((Zt.delta_vs_zero_shot > 0) & (Zt.p < 0.05)).sum()} of {len(Zt)} cells\n")
Zt.pivot_table(index=["scale", "dataset"], columns="mechanism", values="delta_vs_zero_shot").round(3)

cells beating a zero-shot prompt (uncorrected):
scale        dataset           mechanism  delta_vs_zero_shot  seeds_above      p
  70B brfss_diabetes    feature_coverage              0.0639            7 0.0156
  70B brfss_diabetes importance_weighted              0.0480            6 0.0391

2 of 24 cells



mechanism             feature_coverage  importance_weighted  random  similarity
scale dataset                                                                  
70B   acsincome                  0.005               -0.004  -0.001      -0.019
      anes                      -0.027               -0.009  -0.009      -0.076
      brfss_diabetes             0.064                0.048   0.058       0.036
8B    acsincome                 -0.006               -0.022  -0.031       0.010
      anes                      -0.047               -0.026  -0.026      -0.097
      brfss_diabetes            -0.023               -0.065  -0.067       0.041

**Only 2 of 24 cells beat using no demonstrations at all** — and neither is
`similarity`. (Zero-shot ran at a single seed, so these are one-sample tests
against a constant that ignore zero-shot's own query-draw noise; weaker than the
paired tests above, and not corrected alongside them.)

The resolution is that **random-k is a weak baseline** — at 8B it often sits
*below* zero-shot. `similarity` genuinely beats it, robustly, with the mechanism
identified. That is not the same as being worth doing:

> Query-conditional demonstration selection recovers a feature-channel gain over
> random demonstrations that survives complete label corruption, at 8B, on all
> three datasets tested. It does **not** make an 8B model better than the same
> model given no demonstrations, and at 70B it is actively harmful.

## Step 6: Ranking improves; the threshold still does not

In [10]:
bb = M[M.mechanism == "random"].set_index(["scale", "dataset", "corruption", "seed"]).bacc
brows = []
for (sc, ds, corr), g in M[M.mechanism == "similarity"].groupby(["scale", "dataset", "corruption"]):
    g = g.sort_values("seed")
    b = bb.loc[(sc, ds, corr)].reindex(g.seed.values).to_numpy()
    a = g.bacc.to_numpy()
    brows.append(dict(scale=sc, dataset=ds, corruption=corr, delta_bacc=(a - b).mean(),
                      p=stats.wilcoxon(a, b).pvalue))
pd.DataFrame(brows).pivot_table(index=["scale", "dataset"], columns="corruption",
                                values=["delta_bacc", "p"]).round(3)

delta_bacc             p       
corruption                  0.0    1.0    0.0    1.0
scale dataset                                       
70B   acsincome           0.022 -0.027  0.039  0.055
      anes               -0.068 -0.070  0.008  0.016
      brfss_diabetes      0.000 -0.109  0.945  0.008
8B    acsincome          -0.009 -0.057  0.742  0.016
      anes               -0.058  0.027  0.008  0.547
      brfss_diabetes      0.041  0.002  0.016  0.945

The Step 2 AUROC gains only partly reach thresholded accuracy: BRFSS 8B
ΔAUROC +0.108 becomes Δbalanced-accuracy +0.041; ACS Income 8B +0.041 becomes
−0.009 (n.s.). Consistent with Notebook 13 §5 — demonstration *selection*
improves the ranking without fixing the decision threshold, so any practical gain
still needs a separate calibration step.

## What this means for SATA (RQ4)

The case for a learned selector is **stronger and narrower** than before:

- There **is** a channel to exploit, now demonstrated rather than assumed, and it
  is feature-based — which is what a learned scorer over feature representations
  is architecturally suited to.
- It is an **8B-scale** phenomenon. Train and evaluate at 8B; a 70B evaluation is
  a falsification test, not the headline.
- The comparison that matters is **SATA vs `similarity`**, not SATA vs random-k.
  `similarity` is the strong baseline now, and the Step 4 diversity hypothesis is
  the specific thing a learned selector could beat it on.
- Every SATA arm must run at **0% and 100% corruption**. A selector that only
  helps at 0% has learned to exploit the label channel Notebook 13 shows is
  inert — i.e. it overfit the training signal.
- **Report against zero-shot.** A selector that beats `similarity` but not
  zero-shot is a negative result for the thesis question, however good the
  ranking metric looks.
- Use **>= 11 seeds** if the evaluation tests a comparable grid (Step 3).

The `+1.5` counter-spurious bonus in `src/models/sata_targets.py:46` remains
undecided and is probably moot: `counter_spurious` falls below chance in Notebook
10 and was never promoted to a GPU arm.